# Transfer Learning — gatos e cachorros

Execute no Google Colab, preferencialmente com GPU, na ordem das células.
Reutilizamos MobileNetV2/ImageNet e treinamos um classificador binário.
Os resultados serão produzidos pela sua execução; o notebook não contém treinamento prévio.


In [ ]:
# Instalar dependencias e reparar os arquivos do TFDS sem reinstalar TensorFlow.
%pip install -q "tensorflow-datasets>=4.9,<5" "importlib-resources>=6.5,<8"
%pip install -q --force-reinstall --no-deps "tensorflow-datasets>=4.9,<5"
# Se ja importou TFDS nesta sessao, reinicie a sessao antes de continuar.


In [ ]:
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import importlib_resources  # Dependencia usada na preparacao do dataset.
import tensorflow_datasets as tfds

print("TFDS version:", getattr(tfds, "__version__", "unknown"))
print("TFDS module:", getattr(tfds, "__file__", None))
if not callable(getattr(tfds, "load", None)):
    raise RuntimeError(
        "TFDS foi importado sem a funcao load. Execute a celula de instalacao, "
        "reinicie a sessao do Colab e execute novamente a partir dos imports. "
        "Verifique se existe um arquivo tensorflow_datasets.py ou um pacote local "
        "tensorflow_datasets ocultando a biblioteca instalada. "
        f"Origem: {getattr(tfds, '__file__', None)}; "
        f"caminhos: {list(getattr(tfds, '__path__', []))}"
    )

SEED = 42
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32
INITIAL_EPOCHS = 5
FINE_TUNE_EPOCHS = 5
FINE_TUNE_LAYERS = 30
RUN_FINE_TUNING = True
ARTIFACTS = Path("/content/transfer-learning/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


## Dados e preparação

O TFDS fornece uma única divisão original. Criamos três faixas distintas antes de embaralhar apenas o treino. A normalização será feita dentro do modelo. O teste não participa do treinamento ou da seleção do modelo.


In [ ]:
(raw_train, raw_val, raw_test), info = tfds.load(
    "cats_vs_dogs:4.0.1",
    split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    as_supervised=True,
    with_info=True,
    shuffle_files=False,
)
class_names = info.features["label"].names
print("Classes:", dict(enumerate(class_names)))
print("Tamanhos:", [int(ds.cardinality()) for ds in (raw_train, raw_val, raw_test)])

def resize(image, label):
    image = tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE)
    return image, tf.cast(label, tf.float32)

def prepare(dataset, training=False):
    # Embaralhar antes de redimensionar reduz o uso do buffer com imagens float32.
    if training:
        dataset = dataset.shuffle(1000, seed=SEED, reshuffle_each_iteration=True)
    return (
        dataset.map(resize, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare(raw_train, training=True)
val_ds = prepare(raw_val)
test_ds = prepare(raw_test)


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.1, seed=SEED + 1),
], name="augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(None, None, 3), name="rgb_pixels")
x = tf.keras.layers.Resizing(*IMAGE_SIZE)(inputs)
x = augmentation(x)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)
# training=False mantém as estatísticas de BatchNormalization durante fine-tuning.
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2, seed=SEED)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)

def compile_model(learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    )

def callbacks(filename):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(ARTIFACTS / filename), monitor="val_loss", save_best_only=True
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]

compile_model(1e-3)
model.summary()


## Treinar o classificador e, opcionalmente, ajustar a base

Primeiro, apenas o novo classificador aprende. No fine-tuning, algumas camadas da base também são ajustadas. A recompilação aplica a nova configuração de camadas treináveis. A melhor fase será escolhida pela perda de validação.


In [ ]:
initial_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks("feature-extraction.keras"),
)
histories = {"feature_extraction": initial_history.history}
candidates = {
    "feature-extraction.keras": float(min(initial_history.history["val_loss"]))
}

if RUN_FINE_TUNING:
    # Começar o ajuste a partir da melhor época da primeira fase.
    model.load_weights(ARTIFACTS / "feature-extraction.keras")
    base_model.trainable = True
    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-FINE_TUNE_LAYERS:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True
    compile_model(1e-5)
    fine_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=callbacks("fine-tuning.keras"),
    )
    histories["fine_tuning"] = fine_history.history
    candidates["fine-tuning.keras"] = float(min(fine_history.history["val_loss"]))

best_checkpoint = min(candidates, key=candidates.get)
model = tf.keras.models.load_model(ARTIFACTS / best_checkpoint)
print("Perdas de validação:", candidates)
print("Modelo selecionado:", best_checkpoint)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Acurácia", "Perda"]):
    train_values = [v for h in histories.values() for v in h[metric]]
    val_values = [v for h in histories.values() for v in h["val_" + metric]]
    epochs = range(1, len(train_values) + 1)
    ax.plot(epochs, train_values, label="Treino")
    ax.plot(epochs, val_values, label="Validação")
    if RUN_FINE_TUNING:
        ax.axvline(len(initial_history.history[metric]) + 0.5,
                   color="gray", linestyle="--", label="Início do fine-tuning")
    ax.set(title=title, xlabel="Época")
    ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACTS / "curvas.png")
plt.show()


## Avaliação final

O limiar de classificação é 0,5. Observe a matriz de confusão e os erros, além da acurácia. Evite ajustar o modelo a partir destes resultados de teste.


In [ ]:
test_metrics = model.evaluate(test_ds, return_dict=True)
labels, probabilities = [], []
for images, batch_labels in test_ds:
    labels.append(batch_labels.numpy().astype(np.int32))
    probabilities.append(model(images, training=False).numpy().ravel())
y_true = np.concatenate(labels)
y_prob = np.concatenate(probabilities)
y_pred = (y_prob >= 0.5).astype(np.int32)
matrix = tf.math.confusion_matrix(y_true, y_pred, num_classes=2).numpy()

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap="Blues")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=class_names,
       yticklabels=class_names, xlabel="Previsto", ylabel="Real",
       title="Matriz de confusão — teste")
for row in range(2):
    for col in range(2):
        ax.text(col, row, str(matrix[row, col]), ha="center", va="center")
fig.tight_layout()
fig.savefig(ARTIFACTS / "matriz-confusao.png")
plt.show()

sample_images, sample_labels = next(iter(test_ds))
sample_probs = model(sample_images[:9], training=False).numpy().ravel()
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, image, label, prob in zip(
    axes.flat, sample_images[:9], sample_labels[:9], sample_probs
):
    predicted = int(prob >= 0.5)
    ax.imshow(image.numpy().clip(0, 255).astype(np.uint8))
    ax.set_title(f"Real: {class_names[int(label)]}\n"
                 f"Previsto: {class_names[predicted]} | P({class_names[1]}): {prob:.2f}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(ARTIFACTS / "previsoes.png")
plt.show()


In [ ]:
model.save(ARTIFACTS / "modelo-final.keras")
report = {
    "dataset": info.full_name,
    "classes": class_names,
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "split": ["train[:80%]", "train[80%:90%]", "train[90%:]"],
    "initial_epochs": INITIAL_EPOCHS,
    "fine_tune_epochs": FINE_TUNE_EPOCHS,
    "fine_tune_layers": FINE_TUNE_LAYERS,
    "learning_rates": {"feature_extraction": 1e-3, "fine_tuning": 1e-5},
    "run_fine_tuning": RUN_FINE_TUNING,
    "selected_checkpoint": best_checkpoint,
    "validation_loss_by_checkpoint": candidates,
    "test_metrics": {key: float(value) for key, value in test_metrics.items()},
    "confusion_matrix": matrix.tolist(),
    "versions": {"tensorflow": tf.__version__, "tfds": tfds.__version__,
                 "numpy": np.__version__},
}
(ARTIFACTS / "resultados.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
(ARTIFACTS / "historico.json").write_text(
    json.dumps(histories, indent=2), encoding="utf-8"
)
archive = shutil.make_archive(
    "/content/transfer-learning-artefatos", "zip", root_dir=ARTIFACTS
)
from google.colab import files
files.download(archive)


## Sua análise

Registre no README do desafio as métricas, o efeito do fine-tuning e os erros observados. Compare as curvas de treino e validação para investigar overfitting.

Referências: [TensorFlow Transfer Learning](https://www.tensorflow.org/tutorials/images/transfer_learning) e [Cats vs Dogs](https://www.tensorflow.org/datasets/catalog/cats_vs_dogs).
